In [4]:
# ============================================================
# STEP 1: CREATE SPARK SESSION
# ============================================================

from pyspark.sql import SparkSession

# Create a lightweight Spark session.
# We use fewer resources because the local computer was lagging.

spark = (
    SparkSession.builder
    .appName("Delivery Time Prediction")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.driver.memory", "512m")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Spark version: 3.1.1
Spark master: spark://spark-master:7077


In [5]:
# ============================================================
# STEP 2: TEST SPARK
# ============================================================

# Create a tiny DataFrame to confirm that
# Spark can execute a distributed operation.

test_df = spark.range(1)

# Count the records
test_count = test_df.count()

print("Spark test result:", test_count)

Spark test result: 1


In [6]:
# ============================================================
# STEP 3: DEFINE HDFS PATHS
# ============================================================

# Raw CSV dataset stored in HDFS
raw_path = "hdfs://namenode:8020/data/delivery/raw/Food_Delivery_Times.csv"

# Cleaned Parquet dataset stored in HDFS
processed_path = "hdfs://namenode:8020/data/delivery/processed"

# Prediction results will be stored here
results_path = "hdfs://namenode:8020/data/delivery/results"

print("Raw path:", raw_path)
print("Processed path:", processed_path)
print("Results path:", results_path)

Raw path: hdfs://namenode:8020/data/delivery/raw/Food_Delivery_Times.csv
Processed path: hdfs://namenode:8020/data/delivery/processed
Results path: hdfs://namenode:8020/data/delivery/results


In [7]:
# ============================================================
# STEP 4: READ CSV DATASET FROM HDFS
# ============================================================

# Read the CSV dataset from HDFS.
# header=True means the first row contains column names.
# inferSchema=True allows Spark to detect numeric columns.

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(raw_path)
)

print("CSV dataset loaded successfully.")

CSV dataset loaded successfully.


In [8]:
# ============================================================
# STEP 5: CHECK RECORD COUNT
# ============================================================

print("Total records:", df_raw.count())

Total records: 1000


In [9]:
# ============================================================
# STEP 6: DISPLAY SAMPLE RECORDS
# ============================================================

# Display the first 10 records.
df_raw.show(10)

+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+
|Order_ID|Distance_km|Weather|Traffic_Level|Time_of_Day|Vehicle_Type|Preparation_Time_min|Courier_Experience_yrs|Delivery_Time_min|
+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+
|     522|       7.93|  Windy|          Low|  Afternoon|     Scooter|                  12|                   1.0|               43|
|     738|      16.42|  Clear|       Medium|    Evening|        Bike|                  20|                   2.0|               84|
|     741|       9.52|  Foggy|          Low|      Night|     Scooter|                  28|                   1.0|               59|
|     661|       7.44|  Rainy|       Medium|  Afternoon|     Scooter|                   5|                   1.0|               37|
|     412|      19.03|  Clear|          Low|    Morning|        Bike|       

In [10]:
# ============================================================
# STEP 7: CHECK DATA TYPES
# ============================================================

df_raw.printSchema()

root
 |-- Order_ID: integer (nullable = true)
 |-- Distance_km: double (nullable = true)
 |-- Weather: string (nullable = true)
 |-- Traffic_Level: string (nullable = true)
 |-- Time_of_Day: string (nullable = true)
 |-- Vehicle_Type: string (nullable = true)
 |-- Preparation_Time_min: integer (nullable = true)
 |-- Courier_Experience_yrs: double (nullable = true)
 |-- Delivery_Time_min: integer (nullable = true)



In [11]:
# ============================================================
# STEP 8: STANDARDIZE COLUMN NAMES
# ============================================================

df = df_raw.toDF(
    "order_id",
    "distance_km",
    "weather",
    "traffic_level",
    "time_of_day",
    "vehicle_type",
    "preparation_time_min",
    "courier_experience_yrs",
    "delivery_time_min"
)

# Display the new column names
print(df.columns)

['order_id', 'distance_km', 'weather', 'traffic_level', 'time_of_day', 'vehicle_type', 'preparation_time_min', 'courier_experience_yrs', 'delivery_time_min']


In [12]:
# ============================================================
# STEP 9: CHECK MISSING VALUES
# ============================================================

from pyspark.sql import functions as F

# Count NULL values in every column.
missing_values = df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in df.columns
])

missing_values.show()

+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+
|order_id|distance_km|weather|traffic_level|time_of_day|vehicle_type|preparation_time_min|courier_experience_yrs|delivery_time_min|
+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+
|       0|          0|     30|           30|         30|           0|                   0|                    30|                0|
+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+



In [13]:
# ============================================================
# STEP 10: CHECK DUPLICATE RECORDS
# ============================================================

# Count duplicate rows.
duplicate_count = (
    df.count() - df.dropDuplicates().count()
)

print("Duplicate records:", duplicate_count)

Duplicate records: 0


In [14]:
# ============================================================
# STEP 11: CALCULATE MEAN COURIER EXPERIENCE
# ============================================================

experience_mean = (
    df.select(
        F.avg("courier_experience_yrs")
    ).first()[0]
)

print("Mean courier experience:", experience_mean)

Mean courier experience: 4.579381443298969


In [15]:
# ============================================================
# STEP 12: HANDLE MISSING VALUES
# ============================================================

# Replace missing categorical values with "Unknown".
# Replace missing courier experience with the calculated mean.

df_clean = df.fillna({
    "weather": "Unknown",
    "traffic_level": "Unknown",
    "time_of_day": "Unknown",
    "courier_experience_yrs": experience_mean
})

print("Missing values handled successfully.")

Missing values handled successfully.


In [16]:
# ============================================================
# STEP 13: VERIFY CLEANED DATA
# ============================================================

# Check NULL values again.
df_clean.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in df_clean.columns
]).show()

+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+
|order_id|distance_km|weather|traffic_level|time_of_day|vehicle_type|preparation_time_min|courier_experience_yrs|delivery_time_min|
+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+
|       0|          0|      0|            0|          0|           0|                   0|                     0|                0|
+--------+-----------+-------+-------------+-----------+------------+--------------------+----------------------+-----------------+



In [17]:
# ============================================================
# STEP 14: CREATE SPARK SQL VIEW
# ============================================================

# Register the cleaned DataFrame as a temporary SQL table.
df_clean.createOrReplaceTempView("delivery")

print("Temporary view 'delivery' created.")

Temporary view 'delivery' created.


In [18]:
# ============================================================
# STEP 15: SAVE CLEANED DATA AS PARQUET
# ============================================================

# Parquet is efficient for Spark analytics.
# The cleaned dataset will be stored in HDFS.

df_clean.write \
    .mode("overwrite") \
    .parquet(processed_path)

print("Cleaned data saved to:", processed_path)

Cleaned data saved to: hdfs://namenode:8020/data/delivery/processed


In [19]:
# ============================================================
# STEP 16: READ PROCESSED PARQUET DATA
# ============================================================

model_df = spark.read.parquet(processed_path)

print("Processed records:", model_df.count())

Processed records: 1000


In [20]:
# ============================================================
# STEP 17: DELIVERY TIME BY WEATHER
# ============================================================

weather_analysis = (
    model_df
    .groupBy("weather")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("delivery_time_min"), 2)
         .alias("average_delivery_time"),
        F.min("delivery_time_min").alias("minimum_delivery_time"),
        F.max("delivery_time_min").alias("maximum_delivery_time")
    )
    .orderBy(F.desc("average_delivery_time"))
)

weather_analysis.show()

+-------+------------+---------------------+---------------------+---------------------+
|weather|total_orders|average_delivery_time|minimum_delivery_time|maximum_delivery_time|
+-------+------------+---------------------+---------------------+---------------------+
|  Snowy|          97|                67.11|                   24|                  115|
|  Rainy|         204|                59.79|                   15|                  141|
|  Foggy|         103|                59.47|                   25|                  116|
|  Windy|          96|                55.46|                   16|                  122|
|Unknown|          30|                 54.2|                   17|                  109|
|  Clear|         470|                53.08|                    8|                  153|
+-------+------------+---------------------+---------------------+---------------------+



In [21]:
# ============================================================
# STEP 18: DELIVERY TIME BY VEHICLE TYPE
# ============================================================

vehicle_analysis = (
    model_df
    .groupBy("vehicle_type")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("delivery_time_min"), 2)
         .alias("average_delivery_time")
    )
    .orderBy(F.desc("average_delivery_time"))
)

vehicle_analysis.show()

+------------+------------+---------------------+
|vehicle_type|total_orders|average_delivery_time|
+------------+------------+---------------------+
|         Car|         195|                 58.2|
|        Bike|         503|                56.57|
|     Scooter|         302|                56.05|
+------------+------------+---------------------+



In [22]:
# ============================================================
# STEP 19: DELIVERY TIME BY TIME OF DAY
# ============================================================

time_analysis = (
    model_df
    .groupBy("time_of_day")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("delivery_time_min"), 2)
         .alias("average_delivery_time")
    )
    .orderBy(F.desc("average_delivery_time"))
)

time_analysis.show()

+-----------+------------+---------------------+
|time_of_day|total_orders|average_delivery_time|
+-----------+------------+---------------------+
|    Unknown|          30|                66.17|
|    Evening|         293|                57.48|
|    Morning|         308|                56.12|
|  Afternoon|         284|                56.08|
|      Night|          85|                55.21|
+-----------+------------+---------------------+



In [23]:
# ============================================================
# STEP 20: DELIVERY TIME BY TRAFFIC LEVEL
# ============================================================

traffic_analysis = (
    model_df
    .groupBy("traffic_level")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("delivery_time_min"), 2)
         .alias("average_delivery_time")
    )
    .orderBy(F.desc("average_delivery_time"))
)

traffic_analysis.show()

+-------------+------------+---------------------+
|traffic_level|total_orders|average_delivery_time|
+-------------+------------+---------------------+
|         High|         197|                64.81|
|      Unknown|          30|                62.07|
|       Medium|         390|                56.02|
|          Low|         383|                52.89|
+-------------+------------+---------------------+



In [24]:
# ============================================================
# STEP 21: CREATE DISTANCE CATEGORIES
# ============================================================

distance_df = model_df.withColumn(
    "distance_category",
    F.when(F.col("distance_km") < 5, "Short")
     .when(F.col("distance_km") < 10, "Medium")
     .when(F.col("distance_km") < 15, "Long")
     .otherwise("Very Long")
)

distance_df.select(
    "distance_km",
    "distance_category",
    "delivery_time_min"
).show(10)

+-----------+-----------------+-----------------+
|distance_km|distance_category|delivery_time_min|
+-----------+-----------------+-----------------+
|       7.93|           Medium|               43|
|      16.42|        Very Long|               84|
|       9.52|           Medium|               59|
|       7.44|           Medium|               37|
|      19.03|        Very Long|               68|
|       19.4|        Very Long|               57|
|       9.52|           Medium|               49|
|      17.39|        Very Long|               46|
|       1.78|            Short|               35|
|      10.62|             Long|               73|
+-----------+-----------------+-----------------+
only showing top 10 rows



In [25]:
# ============================================================
# STEP 22: DELIVERY TIME BY DISTANCE CATEGORY
# ============================================================

distance_analysis = (
    distance_df
    .groupBy("distance_category")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("distance_km"), 2)
         .alias("average_distance"),
        F.round(F.avg("delivery_time_min"), 2)
         .alias("average_delivery_time")
    )
    .orderBy("average_distance")
)

distance_analysis.show()

+-----------------+------------+----------------+---------------------+
|distance_category|total_orders|average_distance|average_delivery_time|
+-----------------+------------+----------------+---------------------+
|            Short|         248|             2.7|                34.33|
|           Medium|         240|            7.36|                49.04|
|             Long|         261|           12.35|                 63.3|
|        Very Long|         251|           17.53|                 79.4|
+-----------------+------------+----------------+---------------------+



In [26]:
# ============================================================
# STEP 23: DEFINE ML FEATURES
# ============================================================

# Numerical input features
numeric_features = [
    "distance_km",
    "preparation_time_min",
    "courier_experience_yrs"
]

# Categorical input features
categorical_features = [
    "weather",
    "traffic_level",
    "time_of_day",
    "vehicle_type"
]

# Target variable
target = "delivery_time_min"

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target:", target)

Numerical features: ['distance_km', 'preparation_time_min', 'courier_experience_yrs']
Categorical features: ['weather', 'traffic_level', 'time_of_day', 'vehicle_type']
Target: delivery_time_min


In [27]:
# ============================================================
# STEP 24: PREPARE ML DATASET
# ============================================================

ml_df = model_df.select(
    numeric_features
    + categorical_features
    + [target]
)

ml_df.show(5)

+-----------+--------------------+----------------------+-------+-------------+-----------+------------+-----------------+
|distance_km|preparation_time_min|courier_experience_yrs|weather|traffic_level|time_of_day|vehicle_type|delivery_time_min|
+-----------+--------------------+----------------------+-------+-------------+-----------+------------+-----------------+
|       7.93|                  12|                   1.0|  Windy|          Low|  Afternoon|     Scooter|               43|
|      16.42|                  20|                   2.0|  Clear|       Medium|    Evening|        Bike|               84|
|       9.52|                  28|                   1.0|  Foggy|          Low|      Night|     Scooter|               59|
|       7.44|                   5|                   1.0|  Rainy|       Medium|  Afternoon|     Scooter|               37|
|      19.03|                  16|                   5.0|  Clear|          Low|    Morning|        Bike|               68|
+-----------+---

In [28]:
# ============================================================
# STEP 25: STRING INDEXER
# ============================================================

from pyspark.ml.feature import StringIndexer

# Convert categorical text values into numerical indexes.
# Example:
# Bike, Car, Scooter -> 0, 1, 2

indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=column + "_index",
        handleInvalid="keep"
    )
    for column in categorical_features
]

print("Created", len(indexers), "StringIndexers.")

Created 4 StringIndexers.


In [29]:
# ============================================================
# STEP 26: ONE-HOT ENCODING
# ============================================================

from pyspark.ml.feature import OneHotEncoder

encoded_features = [
    column + "_encoded"
    for column in categorical_features
]

encoder = OneHotEncoder(
    inputCols=[
        column + "_index"
        for column in categorical_features
    ],
    outputCols=encoded_features
)

print("Encoded features:", encoded_features)

Encoded features: ['weather_encoded', 'traffic_level_encoded', 'time_of_day_encoded', 'vehicle_type_encoded']


In [30]:
# ============================================================
# STEP 27: VECTOR ASSEMBLER
# ============================================================

from pyspark.ml.feature import VectorAssembler

# Combine numerical and categorical features
# into one Spark ML feature vector.

assembler_inputs = (
    numeric_features +
    encoded_features
)

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

print("Total feature inputs:", len(assembler_inputs))

Total feature inputs: 7


In [31]:
# ============================================================
# STEP 28: TRAIN / TEST SPLIT
# ============================================================

# 80% training data
# 20% testing data

train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training records:", train_df.count())
print("Testing records:", test_df.count())

Training records: 838
Testing records: 162


In [32]:
# ============================================================
# STEP 29: LINEAR REGRESSION MODEL
# ============================================================

from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression

# Create Linear Regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol=target,
    predictionCol="prediction"
)

# Build the complete ML pipeline
lr_pipeline = Pipeline(
    stages=indexers + [encoder, assembler, lr]
)

print("Linear Regression pipeline created.")

Linear Regression pipeline created.


In [33]:
# ============================================================
# STEP 30: TRAIN LINEAR REGRESSION
# ============================================================

# Train the pipeline using training data.
lr_model = lr_pipeline.fit(train_df)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [34]:
# ============================================================
# STEP 31: LINEAR REGRESSION PREDICTIONS
# ============================================================

lr_predictions = lr_model.transform(test_df)

# Display actual and predicted delivery times
lr_predictions.select(
    target,
    "prediction"
).show(10)

+-----------------+------------------+
|delivery_time_min|        prediction|
+-----------------+------------------+
|               16| 15.66126909367254|
|               41| 42.79069943811602|
|               21|23.289493528642836|
|               36| 38.84766266427569|
|               20|22.533031457872376|
|               26| 24.27797365436968|
|               34| 35.23261735175387|
|               30| 30.14146786625963|
|               31| 34.49492587466213|
|               30|36.392347753962014|
+-----------------+------------------+
only showing top 10 rows



In [35]:
# ============================================================
# STEP 32: EVALUATE LINEAR REGRESSION
# ============================================================

from pyspark.ml.evaluation import RegressionEvaluator

# MAE
mae_evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="mae"
)

# RMSE
rmse_evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="rmse"
)

# R-squared
r2_evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="r2"
)

lr_mae = mae_evaluator.evaluate(lr_predictions)
lr_rmse = rmse_evaluator.evaluate(lr_predictions)
lr_r2 = r2_evaluator.evaluate(lr_predictions)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", round(lr_mae, 2))
print("RMSE:", round(lr_rmse, 2))
print("R²  :", round(lr_r2, 2))

Linear Regression Results
-------------------------
MAE : 6.41
RMSE: 10.74
R²  : 0.77


In [36]:
# ============================================================
# STEP 33: RANDOM FOREST REGRESSOR
# ============================================================

from pyspark.ml.regression import RandomForestRegressor

# Create Random Forest regression model.
# A small number of trees keeps the local system lightweight.

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=target,
    predictionCol="prediction",
    numTrees=30,
    maxDepth=5,
    seed=42
)

# Build Random Forest pipeline
rf_pipeline = Pipeline(
    stages=indexers + [encoder, assembler, rf]
)

print("Random Forest pipeline created.")

Random Forest pipeline created.


In [37]:
# ============================================================
# STEP 34: TRAIN RANDOM FOREST
# ============================================================

rf_model = rf_pipeline.fit(train_df)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [38]:
# ============================================================
# STEP 35: RANDOM FOREST PREDICTIONS
# ============================================================

rf_predictions = rf_model.transform(test_df)

rf_predictions.select(
    target,
    "prediction"
).show(10)

+-----------------+------------------+
|delivery_time_min|        prediction|
+-----------------+------------------+
|               16|32.657094902127994|
|               41|44.931003085335334|
|               21| 37.71510380133135|
|               36| 43.83235582998911|
|               20| 37.45151902152769|
|               26| 43.90345104175167|
|               34|44.543153509268556|
|               30| 41.42159818259214|
|               31| 41.47979306204047|
|               30| 46.84370113266352|
+-----------------+------------------+
only showing top 10 rows



In [39]:
# ============================================================
# STEP 36: EVALUATE RANDOM FOREST
# ============================================================

rf_mae = mae_evaluator.evaluate(rf_predictions)
rf_rmse = rmse_evaluator.evaluate(rf_predictions)
rf_r2 = r2_evaluator.evaluate(rf_predictions)

print("Random Forest Results")
print("---------------------")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 2))

Random Forest Results
---------------------
MAE : 8.9
RMSE: 13.22
R²  : 0.65


In [40]:
# ============================================================
# STEP 37: MODEL COMPARISON
# ============================================================

print("========================================")
print("MODEL COMPARISON")
print("========================================")

print("\nLinear Regression")
print("MAE :", round(lr_mae, 2))
print("RMSE:", round(lr_rmse, 2))
print("R²  :", round(lr_r2, 2))

print("\nRandom Forest")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 2))

MODEL COMPARISON

Linear Regression
MAE : 6.41
RMSE: 10.74
R²  : 0.77

Random Forest
MAE : 8.9
RMSE: 13.22
R²  : 0.65


In [41]:
# ============================================================
# STEP 38: SELECT BEST MODEL
# ============================================================

# Select the model with the lower RMSE.

if rf_rmse < lr_rmse:
    best_model = rf_model
    best_model_name = "Random Forest"
    best_predictions = rf_predictions
else:
    best_model = lr_model
    best_model_name = "Linear Regression"
    best_predictions = lr_predictions

print("Best model:", best_model_name)

Best model: Linear Regression


In [42]:
# ============================================================
# STEP 39: PREPARE PREDICTION RESULTS
# ============================================================

prediction_results = best_predictions.select(
    target,
    "prediction"
)

prediction_results.show(10)

+-----------------+------------------+
|delivery_time_min|        prediction|
+-----------------+------------------+
|               16| 15.66126909367254|
|               41| 42.79069943811602|
|               21|23.289493528642836|
|               36| 38.84766266427569|
|               20|22.533031457872376|
|               26| 24.27797365436968|
|               34| 35.23261735175387|
|               30| 30.14146786625963|
|               31| 34.49492587466213|
|               30|36.392347753962014|
+-----------------+------------------+
only showing top 10 rows



In [43]:
# ============================================================
# STEP 40: CALCULATE PREDICTION ERROR
# ============================================================

prediction_results = prediction_results.withColumn(
    "absolute_error",
    F.abs(
        F.col(target) - F.col("prediction")
    )
)

prediction_results.show(10)

+-----------------+------------------+------------------+
|delivery_time_min|        prediction|    absolute_error|
+-----------------+------------------+------------------+
|               16| 15.66126909367254|0.3387309063274593|
|               41| 42.79069943811602|1.7906994381160217|
|               21|23.289493528642836|2.2894935286428364|
|               36| 38.84766266427569|2.8476626642756884|
|               20|22.533031457872376| 2.533031457872376|
|               26| 24.27797365436968|1.7220263456303186|
|               34| 35.23261735175387|1.2326173517538734|
|               30| 30.14146786625963|0.1414678662596316|
|               31| 34.49492587466213|3.4949258746621297|
|               30|36.392347753962014| 6.392347753962014|
+-----------------+------------------+------------------+
only showing top 10 rows



In [44]:
# ============================================================
# STEP 41: AVERAGE ABSOLUTE ERROR
# ============================================================

average_error = (
    prediction_results
    .select(F.avg("absolute_error"))
    .first()[0]
)

print(
    "Average absolute prediction error:",
    round(average_error, 2),
    "minutes"
)

Average absolute prediction error: 6.41 minutes


In [45]:
# ============================================================
# STEP 42: SAVE PREDICTIONS TO HDFS
# ============================================================

# Save the prediction results as Parquet.
# This creates the final analytical output in HDFS.

prediction_results.write \
    .mode("overwrite") \
    .parquet(results_path + "/predictions")

print(
    "Prediction results saved to:",
    results_path + "/predictions"
)

Prediction results saved to: hdfs://namenode:8020/data/delivery/results/predictions


In [46]:
# ============================================================
# STEP 43: VERIFY SAVED RESULTS
# ============================================================

saved_predictions = spark.read.parquet(
    results_path + "/predictions"
)

print(
    "Saved prediction records:",
    saved_predictions.count()
)

saved_predictions.show(10)

Saved prediction records: 162
+-----------------+------------------+------------------+
|delivery_time_min|        prediction|    absolute_error|
+-----------------+------------------+------------------+
|               16| 15.66126909367254|0.3387309063274593|
|               41| 42.79069943811602|1.7906994381160217|
|               21|23.289493528642836|2.2894935286428364|
|               36| 38.84766266427569|2.8476626642756884|
|               20|22.533031457872376| 2.533031457872376|
|               26| 24.27797365436968|1.7220263456303186|
|               34| 35.23261735175387|1.2326173517538734|
|               30| 30.14146786625963|0.1414678662596316|
|               31| 34.49492587466213|3.4949258746621297|
|               30|36.392347753962014| 6.392347753962014|
+-----------------+------------------+------------------+
only showing top 10 rows



In [47]:
# ============================================================
# STEP 44: FINAL MODEL SUMMARY
# ============================================================

print("==============================================")
print("DELIVERY TIME PREDICTION - FINAL SUMMARY")
print("==============================================")

print("Dataset records:", model_df.count())

print("\nLinear Regression:")
print("MAE :", round(lr_mae, 2))
print("RMSE:", round(lr_rmse, 2))
print("R²  :", round(lr_r2, 2))

print("\nRandom Forest:")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 2))

print("\nBest Model:", best_model_name)

print("\nPrediction results:")
print(results_path + "/predictions")

DELIVERY TIME PREDICTION - FINAL SUMMARY
Dataset records: 1000

Linear Regression:
MAE : 6.41
RMSE: 10.74
R²  : 0.77

Random Forest:
MAE : 8.9
RMSE: 13.22
R²  : 0.65

Best Model: Linear Regression

Prediction results:
hdfs://namenode:8020/data/delivery/results/predictions
